In [5]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import numpy as np
import editdistance
import torch.nn.functional as F

In [6]:
base_dir = os.getcwd()

ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_transliteration_train.txt') 

images_dir = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_word_train')

In [7]:
filenames = []
labels = []

# Read the ground truth text file
with open(ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # Ensure the line is not empty
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                filenames.append(filename)
                labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the data
data = pd.DataFrame({
    'filename': filenames,
    'label': labels
})

# Display the DataFrame
data.head()


,filename,label
0,train1.png,kagastu
1,train2.png,","
2,train3.png,gelak
3,train4.png,ancak
4,train5.png,","


In [8]:
# Build a character vocabulary from the labels
all_text = ''.join(data['label'])  # Concatenate all labels into one string
chars = sorted(list(set(all_text)))  # Extract unique characters and sort them

# # Create character to index mapping, starting from index 1
# char_to_idx = {char: idx + 1 for idx, char in enumerate(chars)}  # Start indexing from 1

# # Add special tokens
# char_to_idx['<PAD>'] = 0  # Add padding token at index 0
# char_to_idx['<UNK>'] = len(char_to_idx)  # Add unknown token
# char_to_idx['<SOS>'] = len(char_to_idx)  # Add start-of-sequence token
# char_to_idx['<EOS>'] = len(char_to_idx)  # Add end-of-sequence token

# Overhaul your char_to_idx so that index 0 is the blank:
char_to_idx = {}
char_to_idx['<BLANK>'] = 0
# Optionally keep <PAD> as 1 if you wish:
char_to_idx['<PAD>'] = 1

# Then proceed with actual characters from index 2 onwards:
for i, char in enumerate(chars, start=2):
    char_to_idx[char] = i

# Create index to character mapping
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# Update vocabulary size
vocab_size = len(char_to_idx)

print(f"Vocabulary size: {vocab_size}")


Vocabulary size: 56


In [9]:
# print("Character to Index Mapping:")
# for char, idx in char_to_idx.items():
#     print(f"'{char}': {idx}")


In [10]:
# empty_labels = data[data['label'].str.strip() == '']
# print(f"Number of empty labels in training data: {len(empty_labels)}")

In [11]:
# add padding to all labels to ensure all labels have same length, use index of character to represent the label
# Encode labels as sequences of character indices
max_label_length = max(len(label) for label in data['label']) + 2  # +2 for <SOS> and <EOS>

# def encode_label(label, char_to_idx, max_length):
#     encoded = (
#         [char_to_idx['<SOS>']] +
#         [char_to_idx.get(char, char_to_idx['<UNK>']) for char in label] +
#         [char_to_idx['<EOS>']]
#     )
#     padded = encoded + [char_to_idx['<PAD>']] * (max_length - len(encoded))
#     return padded

def encode_label_ctc(label, char_to_idx, max_length):
    # Convert each character to index
    encoded = [char_to_idx.get(char, char_to_idx['<UNK>']) for char in label]
    # Pad if needed
    if len(encoded) < max_length:
        encoded += [char_to_idx['<PAD>']] * (max_length - len(encoded))
    else:
        encoded = encoded[:max_length]
    return encoded


data['encoded_label'] = data['label'].apply(lambda x: encode_label(x, char_to_idx, max_label_length))

NameError: name 'encode_label' is not defined

In [ ]:
# Split the dataset into training and validation sets
train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)

In [ ]:
# Define a custom dataset class
# Image data
class BalineseDataset(Dataset):
    def __init__(self, data, images_dir, transform=None):
        self.data = data.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """given index, return corresponding image and label"""
        img_name = self.data.loc[idx, 'filename']
        label = self.data.loc[idx, 'encoded_label']
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label


In [ ]:
# Define image transformations
transform = transforms.Compose([
    transforms.Resize((64, 256)),  # Resize images to a fixed size
    transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5),  # Mean for R, G, B channels
                            (0.5, 0.5, 0.5))  # Std for R, G, B channels  # Normalize images
])

# Create dataset instances
train_dataset = BalineseDataset(train_data, images_dir, transform=transform)
val_dataset = BalineseDataset(val_data, images_dir, transform=transform)


In [ ]:
# Create data loaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_dim=256, num_lstm_layers=2):
        super(CRNN, self).__init__()
        # 1) CNN (can be a smaller backbone or ResNet w/ modifications)
        resnet = models.resnet18(pretrained=True)
        layers = list(resnet.children())[:-2]  # remove avgpool & fc
        self.cnn = nn.Sequential(*layers)
        
        # we need a way to adapt the 512-dim output (from resnet) to a 
        # sequential representation along the width. Let's do AdaptiveAvgPool2d for height = 1 
        # so we end up with shape (batch, 512, 1, width//32).
        self.pool = nn.AdaptiveAvgPool2d((1, None))  # keep width dimension variable, compress height to 1

        # 2) BiLSTM
        # We'll find out the feature dimension from the CNN. If the width is 256 -> after /32 => 8 
        # but let's keep it flexible with self.pool
        self.lstm = nn.LSTM(
            input_size=512,  # from the CNN
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        # 3) Final linear layer: map to num_classes (including blank)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # bidirectional => *2

    def forward(self, x):
        # x shape: (batch, 3, height=64, width=256) typically
        features = self.cnn(x)  # -> (batch, 512, h', w')
        features = self.pool(features)  # -> (batch, 512, 1, w'')
        # Now shape is (batch, 512, 1, w'')
        features = features.squeeze(2)  # -> (batch, 512, w'')
        features = features.permute(0, 2, 1)  # -> (batch, w'', 512)

        # pass to LSTM
        lstm_out, _ = self.lstm(features)  # (batch, seq_len, hidden_size*2)

        # map to logits
        logits = self.fc(lstm_out)  # shape: (batch, seq_len, num_classes)

        # for CTC, we want shape: (seq_len, batch, num_classes), so we permute
        logits = logits.permute(1, 0, 2)  # (seq_len, batch, num_classes)
        return logits

In [ ]:
num_classes = vocab_size  # including blank
model = CRNN(num_classes=num_classes)
model = model.to(device)

ctc_loss_fn = nn.CTCLoss(blank=0, reduction='mean')  # 0 as blank index
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    # 1) Forward
    logits = model(images)  
    # logits shape: (seq_len, batch, num_classes)

    # 2) Compute input_lengths
    # seq_len = logits.shape[0], for every sample in the batch.
    input_lengths = torch.full(size=(logits.size(1),), fill_value=logits.size(0), dtype=torch.long)

    # 3) Compute target_lengths and flatten labels
    target_list = []
    lengths_list = []
    for i in range(labels.size(0)):
        # remove any <PAD> if you used it
        raw = labels[i].tolist()
        # remove trailing <PAD> (or anything not in real char set)
        cleaned = [r for r in raw if r not in [1]]  # if 1 == <PAD>
        target_list.extend(cleaned)
        lengths_list.append(len(cleaned))

    targets = torch.tensor(target_list, dtype=torch.long)
    target_lengths = torch.tensor(lengths_list, dtype=torch.long)

    # 4) CTC loss
    loss = ctc_loss_fn(logits, targets, input_lengths, target_lengths)

    # 5) Backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
epoch = 2
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # 1) Forward
        logits = model(images)  # => (seq_len, batch, num_classes)
        seq_len = logits.size(0)
        batch_size = logits.size(1)

        # 2) input_lengths: how many timesteps for each sample in the batch
        input_lengths = torch.full(
            size=(batch_size,),
            fill_value=seq_len,
            dtype=torch.long
        ).to(device)

        # 3) Flatten out the targets & compute target_lengths
        target_list = []
        lengths_list = []
        for i in range(batch_size):
            raw = labels[i].cpu().tolist()
            # remove <PAD> (or any special if needed)
            cleaned = [r for r in raw if r != char_to_idx['<PAD>']]
            target_list.extend(cleaned)
            lengths_list.append(len(cleaned))

        targets = torch.tensor(target_list, dtype=torch.long).to(device)
        target_lengths = torch.tensor(lengths_list, dtype=torch.long).to(device)

        # 4) Compute CTC loss
        loss = ctc_loss_fn(logits, targets, input_lengths, target_lengths)

        # 5) Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")


In [ ]:
model.eval()
total_cer = 0.0
count = 0
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)  # => (seq_len, batch, num_classes)

        # same input_lengths logic as train
        seq_len, batch_size, _ = logits.size()
        input_lengths = torch.full(
            (batch_size,), seq_len, dtype=torch.long
        ).to(device)

        # flatten labels
        target_list = []
        lengths_list = []
        for i in range(batch_size):
            raw = labels[i].cpu().tolist()
            cleaned = [r for r in raw if r != char_to_idx['<PAD>']]
            target_list.extend(cleaned)
            lengths_list.append(len(cleaned))

        targets = torch.tensor(target_list, dtype=torch.long).to(device)
        target_lengths = torch.tensor(lengths_list, dtype=torch.long).to(device)

        loss = ctc_loss_fn(logits, targets, input_lengths, target_lengths)

        # decode
        pred_indices = ctc_greedy_decode(logits, blank=char_to_idx['<BLANK>'] if '<BLANK>' in char_to_idx else 0)

        # compute CER
        for b in range(batch_size):
            # predicted
            pred_str = ''.join(idx_to_char[idx] for idx in pred_indices[b])

            # ground truth
            raw_gt = [r for r in labels[b].cpu().tolist() if r != char_to_idx['<PAD>']]
            gt_str = ''.join(idx_to_char[idx] for idx in raw_gt)

            # compute CER
            if len(gt_str) > 0:
                cer = editdistance.eval(pred_str, gt_str) / len(gt_str)
                total_cer += cer
            count += 1

print(f"Validation CER: {total_cer / count}")


In [ ]:
def ctc_greedy_decode(logits, blank=0):
    # logits: (seq_len, batch, num_classes)
    pred_labels = []
    seq_len, batch_size, num_classes = logits.size()
    softmaxed = torch.log_softmax(logits, dim=2)  # or use F.softmax

    max_probs = softmaxed.argmax(dim=2)  # (seq_len, batch)
    
    for b in range(batch_size):
        raw_sequence = max_probs[:, b].tolist()  # length seq_len
        # Remove consecutive duplicates & blank
        deduped = []
        prev = None
        for char_idx in raw_sequence:
            if char_idx != blank and char_idx != prev:
                deduped.append(char_idx)
            prev = char_idx
        pred_labels.append(deduped)
    return pred_labels


In [ ]:
# Paths to test data
test_ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_transliteration_test.txt')
test_images_dir = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_word_test')

test_filenames = []
test_labels = []

# Read the ground truth text file for the test set
with open(test_ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                test_filenames.append(filename)
                test_labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the test data
test_data = pd.DataFrame({
    'filename': test_filenames,
    'label': test_labels
})


In [ ]:
max_label_length_test = max(len(label) for label in test_data['label']) + 2


def encode_label_test(label, char_to_idx, max_length):
    encoded = [char_to_idx['<SOS>']] + [char_to_idx.get(char, char_to_idx['<UNK>']) for char in label] + [char_to_idx['<EOS>']]
    if len(encoded) > max_length:
        encoded = encoded[:max_length]
    else:
        encoded += [char_to_idx['<PAD>']] * (max_length - len(encoded))
    return encoded


# Apply the encoding to the test data
test_data['encoded_label'] = test_data['label'].apply(lambda x: encode_label(x, char_to_idx, max_label_length_test))


In [ ]:
# Define the same transformations used during training
test_transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Create test dataset
test_dataset = BalineseDataset(test_data, test_images_dir, transform=test_transform)

# Create test data loader
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


In [ ]:
def inference(encoder, decoder, dataloader, device, char_to_idx, idx_to_char, max_seq_length):
    encoder.eval()
    decoder.eval()
    results = []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(dataloader):
            images = images.to(device)
            batch_size = images.size(0)

            # Forward pass through encoder
            encoder_out = encoder(images)
            encoder_dim = encoder_out.size(-1)
            num_pixels = encoder_out.size(1)

            # Initialize LSTM state
            h, c = decoder.init_hidden_state(encoder_out)

            # Start with <SOS> token
            inputs = torch.full((batch_size,), char_to_idx['<SOS>'], dtype=torch.long).to(device)

            # Store predicted sequences
            predicted_seqs = []

            for _ in range(max_seq_length):
                embeddings = decoder.embedding(inputs)

                attention_weighted_encoding, alpha = decoder.attention(encoder_out, h)

                gate = decoder.sigmoid(decoder.f_beta(h))
                attention_weighted_encoding = gate * attention_weighted_encoding

                h, c = decoder.decode_step(
                    torch.cat([embeddings, attention_weighted_encoding], dim=1),
                    (h, c)
                )

                preds = decoder.fc(decoder.dropout(h))
                _, preds_idx = preds.max(1)

                predicted_seqs.append(preds_idx.cpu().numpy())

                inputs = preds_idx

            # Convert predictions to text
            predicted_seqs = np.array(predicted_seqs).T

            for i in range(batch_size):
                pred_indices = predicted_seqs[i]

                # Stop at <EOS>
                if char_to_idx['<EOS>'] in pred_indices:
                    eos_index = np.where(pred_indices == char_to_idx['<EOS>'])[0][0]
                    pred_indices = pred_indices[:eos_index]
                else:
                    pred_indices = pred_indices

                # Decode indices to characters
                pred_chars = [idx_to_char.get(idx, '') for idx in pred_indices]

                # Join characters to form strings
                pred_str = ''.join(pred_chars)

                # Get ground truth label (remove <SOS> and <EOS>)
                label_indices = labels[i].cpu().numpy()
                label_indices = label_indices[1:]  # Remove <SOS>
                if char_to_idx['<EOS>'] in label_indices:
                    eos_index = np.where(label_indices == char_to_idx['<EOS>'])[0][0]
                    label_indices = label_indices[:eos_index]
                else:
                    label_indices = label_indices[label_indices != char_to_idx['<PAD>']]

                label_chars = [idx_to_char.get(idx, '') for idx in label_indices]
                label_str = ''.join(label_chars)

                # Collect results
                image_filename = test_data.iloc[batch_idx * batch_size + i]['filename']
                results.append({
                    'image_filename': image_filename,
                    'predicted_caption': pred_str,
                    'ground_truth_caption': label_str
                })

    return results

In [ ]:
# Run inference on the test set
test_results = inference(encoder, decoder, test_loader, device, char_to_idx, idx_to_char, max_label_length_test)


In [ ]:
def calculate_cer(reference, hypothesis):
    if len(reference) == 0:
        return float('inf') if len(hypothesis) > 0 else 0.0
    return editdistance.eval(reference, hypothesis) / len(reference)

# Compute CER for each sample
cer_scores = []
for result in test_results:
    cer = calculate_cer(result['ground_truth_caption'], result['predicted_caption'])
    cer_scores.append(cer)


# Initialize total edit distance and total reference length
total_edit_distance = 0
total_reference_length = 0

for i, result in enumerate(test_results):
    reference = result['ground_truth_caption']
    hypothesis = result['predicted_caption']
    
    # Calculate edit distance
    edit_distance = editdistance.eval(reference, hypothesis)
    total_edit_distance += edit_distance
    total_reference_length += len(reference)
    
    # Optionally, compute per-sample CER and store it
    if len(reference) > 0:
        cer = edit_distance / len(reference)
    else:
        cer = 0.0  # Avoid division by zero
    test_results[i]['cer'] = cer  # Add CER to each result dictionary if needed

# Compute total CER
if total_reference_length > 0:
    total_cer = total_edit_distance / total_reference_length
else:
    total_cer = 0.0  # Avoid division by zero

print(f"Total CER on Test Set: {total_cer:.4f}")

average_cer = sum(cer_scores) / len(cer_scores)
print(f"Average CER on Test Set: {average_cer:.4f}")



In [12]:
# Compute CER for each sample and add it to test_results
cer_scores = []
for i, result in enumerate(test_results):
    cer = calculate_cer(result['ground_truth_caption'], result['predicted_caption'])
    cer_scores.append(cer)
    test_results[i]['cer'] = cer  # Add CER to each result dictionary

# Compute average CER
average_cer = sum(cer_scores) / len(cer_scores)
print(f"Average CER on Test Set: {average_cer:.4f}")

# Sort test_results by CER in descending order to find samples with highest errors
sorted_results = sorted(test_results, key=lambda x: x['cer'], reverse=True)

# Print top N samples with highest CER
N = 30
print(f"\nTop {N} samples with highest CER:")
for i in range(N):
    result = sorted_results[i]
    print(f"Sample {i + 1}:")
    print(f"Image Filename: {result['image_filename']}")
    print(f"CER: {result['cer']:.4f}")
    print(f"Ground Truth Caption: {result['ground_truth_caption']}")
    print(f"Predicted Caption  : {result['predicted_caption']}\n")


NameError: name 'test_results' is not defined

In [ ]:
# Additionally, print some random samples to get a general idea of model performance
import random

M = 5  # Number of random samples to display
random_samples = random.sample(test_results, M)
print(f"\nRandom {M} samples from Test Set:")
for i, result in enumerate(random_samples):
    print(f"Sample {i + 1}:")
    print(f"Image Filename: {result['image_filename']}")
    print(f"CER: {result['cer']:.4f}")
    print(f"Ground Truth Caption: {result['ground_truth_caption']}")
    print(f"Predicted Caption  : {result['predicted_caption']}\n")